# Phishing Website Detection Using Machine Learning

---

| Field | Details |
|---|---|
| **Student Name** | Sakshi Kumari |
| **Program** | IBM SkillsBuild Cyber Security 2026 |
| **Organization** | BharatCares |
| **Project Domain** | Cybersecurity · Phishing Detection · Machine Learning |
| **Language** | Python 3 |
| **Environment** | Jupyter Notebook |
| **Dataset** | UCI ML Repository — Phishing Websites (ID 327) |

---

## 1. Introduction

Phishing is a form of cybercrime where attackers create fake websites that closely mimic legitimate ones to steal sensitive information such as passwords, credit card numbers, and personal data. According to cybersecurity reports, phishing remains one of the most common and damaging forms of online attack.

This project applies **supervised machine learning** to automatically classify websites as **legitimate** or **phishing** based on a set of URL and web-page features. By training a model on known examples, we aim to build a system that can assist in detecting potentially dangerous websites.

### Project Objectives

1. Load and explore the UCI Phishing Websites Dataset.
2. Understand the meaning of dataset features in a cybersecurity context.
3. Preprocess the data and perform exploratory data analysis.
4. Train a machine learning classifier (Random Forest).
5. Evaluate the model using standard metrics and interpret results.
6. Demonstrate predictions on test samples.
7. Discuss model limitations and responsible use.

> **Disclaimer:** This model produces probabilistic estimates based on known patterns. It can produce **false positives** (flagging a safe site as phishing) and **false negatives** (missing an actual phishing site). It must **not** be used as the sole security measure.

## 2. Dataset Description

**Source:** [UCI Machine Learning Repository — Phishing Websites Dataset](https://archive.ics.uci.edu/dataset/327/phishing+websites)

**Instances:** 11,055 website records  
**Features:** 30 input features + 1 target variable  
**Task:** Binary Classification  

### Feature Encoding

Most features use a 3-value or 2-value categorical encoding:

| Encoded Value | General Meaning |
|---|---|
| `-1` | Phishing indicator (suspicious/bad sign) |
| `0` | Suspicious / neutral indicator |
| `1` | Legitimate indicator (good sign) |

### Target Variable: `Result`

| Value | Meaning |
|---|---|
| `-1` | **Phishing** website |
| `1` | **Legitimate** website |

### Key Features Explained

| Feature | Description |
|---|---|
| `having_IP_Address` | URL uses raw IP instead of domain name — common in phishing |
| `URL_Length` | Very long URLs often hide malicious content |
| `Shortining_Service` | URL shorteners (bit.ly etc.) can mask phishing URLs |
| `having_At_Symbol` | `@` symbol in URL — browser ignores everything before it |
| `SSLfinal_State` | Whether a valid HTTPS/SSL certificate is present |
| `Domain_registeration_length` | Short registration periods suggest phishing |
| `Favicon` | Favicon loaded from a different domain |
| `HTTPS_token` | The word 'HTTPS' used in domain name to fool users |
| `Request_URL` | Percentage of external objects loaded from other domains |
| `URL_of_Anchor` | Anchor tags pointing to different domains |
| `age_of_domain` | Newly registered domains are more likely phishing |
| `DNSRecord` | No DNS record — domain may not be legitimate |
| `web_traffic` | Low traffic ranks indicate less trustworthy sites |
| `Google_Index` | Site not indexed by Google — suspicious |
| `Statistical_report` | Domain/IP listed in phishing statistical reports |

## 3. Import Required Libraries

We import all necessary libraries at the top of the notebook. Run this cell first.

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import os
import warnings
warnings.filterwarnings('ignore')

# ── Data handling ─────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns

# ── Machine learning ──────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

# ── Model persistence ─────────────────────────────────────────────────────────
import joblib

# ── Dataset download ─────────────────────────────────────────────────────────
try:
    from ucimlrepo import fetch_ucirepo
    UCIMLREPO_AVAILABLE = True
except ImportError:
    UCIMLREPO_AVAILABLE = False
    print("[INFO] ucimlrepo not installed. Will use fallback manual loading.")

# ── Reproducibility ───────────────────────────────────────────────────────────
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# ── Plot style ────────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

print("✅ All libraries imported successfully.")
print(f"   NumPy  : {np.__version__}")
print(f"   Pandas : {pd.__version__}")

## 4. Load the Dataset

We use the **`ucimlrepo`** package (UCI's official Python client) to automatically download the Phishing Websites dataset.  

If `ucimlrepo` is not available, a fallback path reads a manually downloaded ARFF file from the `data/` folder.

### Manual Download (if automatic fails)

1. Visit: https://archive.uci.edu/dataset/327/phishing+websites
2. Click **Download** to get `phishing+websites.zip`
3. Extract the ZIP — you will find `Training Dataset.arff`
4. Copy it to the `data/` folder in this project
5. Re-run the cell below — it will load from `data/Training Dataset.arff`

In [ ]:
# Column names for the UCI Phishing Websites dataset (30 features + 1 target)
COLUMN_NAMES = [
    'having_IP_Address', 'URL_Length', 'Shortining_Service',
    'having_At_Symbol', 'double_slash_redirecting', 'Prefix_Suffix',
    'having_Sub_Domain', 'SSLfinal_State', 'Domain_registeration_length',
    'Favicon', 'port', 'HTTPS_token', 'Request_URL', 'URL_of_Anchor',
    'Links_in_tags', 'SFH', 'Submitting_to_email', 'Abnormal_URL',
    'Redirect', 'on_mouseover', 'RightClick', 'popUpWidnow', 'Iframe',
    'age_of_domain', 'DNSRecord', 'web_traffic', 'Page_Rank',
    'Google_Index', 'Links_pointing_to_page', 'Statistical_report',
    'Result'
]

df = None  # Will hold our DataFrame

# ── Method 1: ucimlrepo automatic download ────────────────────────────────────
if UCIMLREPO_AVAILABLE:
    print("Attempting automatic download via ucimlrepo ...")
    try:
        phishing_data = fetch_ucirepo(id=327)
        X_raw = phishing_data.data.features
        y_raw = phishing_data.data.targets
        df = pd.concat([X_raw, y_raw], axis=1)
        # Standardise target column name
        target_col = df.columns[-1]
        if target_col != 'Result':
            df.rename(columns={target_col: 'Result'}, inplace=True)
        print(f"✅ Dataset loaded via ucimlrepo. Shape: {df.shape}")
    except Exception as e:
        print(f"[WARNING] ucimlrepo download failed: {e}")
        df = None

# ── Method 2: Manual ARFF file in data/ folder ────────────────────────────────
if df is None:
    arff_path = os.path.join('data', 'Training Dataset.arff')
    if os.path.exists(arff_path):
        print(f"Loading from local ARFF file: {arff_path}")
        # Parse ARFF manually — skip header lines starting with '@'
        rows = []
        with open(arff_path, 'r') as f:
            in_data = False
            for line in f:
                line = line.strip()
                if line.upper().startswith('@DATA'):
                    in_data = True
                    continue
                if in_data and line and not line.startswith('%'):
                    rows.append(line.split(','))
        df = pd.DataFrame(rows, columns=COLUMN_NAMES)
        df = df.apply(pd.to_numeric, errors='coerce')
        print(f"✅ Dataset loaded from ARFF. Shape: {df.shape}")
    else:
        raise FileNotFoundError(
            "Dataset not found!\n"
            "Please follow the manual download instructions in the cell above "
            "and place 'Training Dataset.arff' in the 'data/' folder."
        )

# Ensure target is integer
df['Result'] = df['Result'].astype(int)
print(f"\nDataset ready. Rows: {df.shape[0]}, Columns: {df.shape[1]}")

## 5. Initial Data Inspection

Before any analysis, let's understand what the data looks like — its size, column names, data types, and a quick peek at values.

In [ ]:
print("=" * 55)
print(" DATASET DIMENSIONS")
print("=" * 55)
print(f"  Rows (website records) : {df.shape[0]:,}")
print(f"  Columns (features + target) : {df.shape[1]}")
print(f"  Input features         : {df.shape[1] - 1}")
print(f"  Target variable        : Result")

In [ ]:
print("First 5 rows of the dataset:")
df.head()

In [ ]:
print("Data types and non-null counts:")
df.info()

In [ ]:
print("Statistical summary of all columns:")
df.describe()

## 6. Data Quality Checks

We check for:
- **Missing values** — cells with NaN/null
- **Duplicate records** — identical rows that could skew training
- **Unexpected values** — values outside the expected {-1, 0, 1} encoding

In [ ]:
# ── Missing values ────────────────────────────────────────────────────────────
missing = df.isnull().sum()
total_missing = missing.sum()

print(f"Total missing values in dataset: {total_missing}")
if total_missing > 0:
    print("\nColumns with missing values:")
    print(missing[missing > 0])
    # Drop rows with any missing values
    df.dropna(inplace=True)
    print(f"\nRows after dropping missing values: {df.shape[0]:,}")
else:
    print("✅ No missing values found.")

In [ ]:
# ── Duplicate records ─────────────────────────────────────────────────────────
n_duplicates = df.duplicated().sum()
print(f"Duplicate rows found: {n_duplicates}")

if n_duplicates > 0:
    df.drop_duplicates(inplace=True)
    df.reset_index(drop=True, inplace=True)
    print(f"Rows after removing duplicates: {df.shape[0]:,}")
else:
    print("✅ No duplicate rows found.")

In [ ]:
# ── Unexpected feature values ─────────────────────────────────────────────────
# Most features should only contain {-1, 0, 1}
# (a few features like Redirect and Page_Rank may have 0 or 2)
feature_cols = [c for c in df.columns if c != 'Result']

unexpected = {}
for col in feature_cols:
    unique_vals = set(df[col].unique())
    outside = unique_vals - {-1, 0, 1, 2, 3}  # allow a few extra categorical levels
    if outside:
        unexpected[col] = outside

if unexpected:
    print("Columns with unexpected values:")
    for col, vals in unexpected.items():
        print(f"  {col}: {vals}")
else:
    print("✅ All feature values are within expected encoded range.")

# Target variable check
print(f"\nUnique values in target 'Result': {sorted(df['Result'].unique())}")
print("Expected: -1 (Phishing) and 1 (Legitimate)")

## 7. Exploratory Data Analysis (EDA)

EDA helps us understand the dataset's patterns, class balance, and feature distributions before we build the model.

In [ ]:
# ── Class Distribution ────────────────────────────────────────────────────────
class_counts = df['Result'].value_counts()
class_labels = {-1: 'Phishing (-1)', 1: 'Legitimate (1)'}

print("Class Distribution:")
print("-" * 30)
for val, count in class_counts.items():
    pct = count / len(df) * 100
    label = class_labels.get(val, str(val))
    print(f"  {label:<20}: {count:>5,}  ({pct:.1f}%)")

print(f"\n  Total records: {len(df):,}")

In [ ]:
# ── Plot 1: Class Distribution Bar Chart ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart
counts_mapped = df['Result'].map({-1: 'Phishing', 1: 'Legitimate'}).value_counts()
colors = ['#e74c3c', '#2ecc71']
axes[0].bar(counts_mapped.index, counts_mapped.values, color=colors, edgecolor='white', width=0.5)
axes[0].set_title('Class Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Website Class')
axes[0].set_ylabel('Number of Records')
for i, (label, val) in enumerate(counts_mapped.items()):
    axes[0].text(i, val + 50, str(val), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(
    counts_mapped.values,
    labels=counts_mapped.index,
    autopct='%1.1f%%',
    colors=colors,
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
axes[1].set_title('Class Distribution (Proportion)', fontsize=14, fontweight='bold')

plt.suptitle('Phishing vs Legitimate Websites', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
os.makedirs('outputs', exist_ok=True)
plt.savefig('outputs/class_distribution.png', bbox_inches='tight', dpi=100)
plt.show()
print("Chart saved to outputs/class_distribution.png")

In [ ]:
# ── Plot 2: Distribution of Key Security Features by Class ────────────────────
key_features = [
    'having_IP_Address', 'SSLfinal_State', 'URL_Length',
    'age_of_domain', 'web_traffic', 'HTTPS_token'
]

df_plot = df.copy()
df_plot['Class'] = df_plot['Result'].map({-1: 'Phishing', 1: 'Legitimate'})

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, feat in enumerate(key_features):
    cross = df_plot.groupby(['Class', feat]).size().unstack(fill_value=0)
    cross.plot(kind='bar', ax=axes[i], colormap='RdYlGn', edgecolor='white')
    axes[i].set_title(f'{feat}', fontsize=11, fontweight='bold')
    axes[i].set_xlabel('Class')
    axes[i].set_ylabel('Count')
    axes[i].tick_params(axis='x', rotation=0)
    axes[i].legend(title='Feature Value', fontsize=8)

plt.suptitle('Key Feature Distributions by Website Class', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/feature_distributions.png', bbox_inches='tight', dpi=100)
plt.show()
print("Chart saved to outputs/feature_distributions.png")

In [ ]:
# ── Plot 3: Correlation Heatmap ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 13))
corr_matrix = df[feature_cols + ['Result']].corr()

mask = np.zeros_like(corr_matrix, dtype=bool)
mask[np.triu_indices_from(mask)] = True  # show lower triangle only

sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=False,
    cmap='coolwarm',
    center=0,
    linewidths=0.3,
    ax=ax,
    cbar_kws={'shrink': 0.8}
)
ax.set_title('Feature Correlation Heatmap', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/correlation_heatmap.png', bbox_inches='tight', dpi=100)
plt.show()
print("Chart saved to outputs/correlation_heatmap.png")

## 8. Feature and Target Separation

We split the dataset into:
- **X** — the 30 input features (URL and web-page attributes)
- **y** — the target variable (`Result`: -1 = Phishing, 1 = Legitimate)

No preprocessing (scaling, encoding) is needed here because:
- All features are already numerically encoded.
- Random Forest is a tree-based algorithm that does not require feature scaling.

In [ ]:
# Separate features from target
X = df.drop(columns=['Result'])
y = df['Result']

print(f"Feature matrix X : {X.shape[0]:,} rows × {X.shape[1]} columns")
print(f"Target vector  y : {y.shape[0]:,} values")
print(f"\nFeature column names:")
for i, col in enumerate(X.columns, 1):
    print(f"  {i:2}. {col}")

## 9. Train / Test Split

We split the data into:
- **80% Training set** — used to train the model
- **20% Test set** — used to evaluate the model on unseen data

We use `stratify=y` to ensure that both sets have the same class proportion as the original dataset. This is important when classes are imbalanced.

> **Why this matters:** If we evaluate on the same data we trained on, the model appears perfect but fails on new websites. Using a separate test set gives an honest estimate of real-world performance.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_SEED,
    stratify=y          # preserve class proportions
)

print("Train / Test Split Results:")
print("-" * 35)
print(f"  Training samples : {len(X_train):,}  ({len(X_train)/len(X)*100:.0f}%)")
print(f"  Testing  samples : {len(X_test):,}  ({len(X_test)/len(X)*100:.0f}%)")
print()
print("Class balance in training set:")
train_dist = y_train.value_counts(normalize=True) * 100
for val, pct in train_dist.items():
    print(f"  {class_labels.get(val, val):<22}: {pct:.1f}%")

print("\nClass balance in test set:")
test_dist = y_test.value_counts(normalize=True) * 100
for val, pct in test_dist.items():
    print(f"  {class_labels.get(val, val):<22}: {pct:.1f}%")

## 10. Model Training — Random Forest Classifier

We use a **Random Forest** classifier, which is an ensemble of many Decision Trees.

### Why Random Forest?

| Advantage | Explanation |
|---|---|
| High accuracy | Combines many trees — reduces overfitting |
| No scaling needed | Works directly with encoded categorical values |
| Handles noise | Robust to outliers and irrelevant features |
| Feature importance | Can tell us which URL features matter most |
| Interpretable | Easy to explain to a non-technical audience |

### How Random Forest Works

1. Build N decision trees, each trained on a random subset of training samples (bootstrapping).
2. Each tree also sees a random subset of features at each split.
3. For prediction, each tree votes — the class with the most votes wins.

This voting process makes Random Forest more accurate and stable than a single Decision Tree.

In [ ]:
# ── Initialise the Random Forest model ────────────────────────────────────────
rf_model = RandomForestClassifier(
    n_estimators=100,       # 100 decision trees in the forest
    max_depth=None,         # Trees grow until all leaves are pure
    min_samples_split=2,    # Minimum samples to split an internal node
    min_samples_leaf=1,     # Minimum samples in a leaf node
    random_state=RANDOM_SEED,
    n_jobs=-1               # Use all available CPU cores
)

print("Training Random Forest on training set ...")
rf_model.fit(X_train, y_train)
print("✅ Model training complete.")
print(f"   Trees in forest : {rf_model.n_estimators}")
print(f"   Features used   : {rf_model.n_features_in_}")

## 11. Model Evaluation

We evaluate the trained model on the **unseen test set**. We use the following metrics:

| Metric | What it Measures |
|---|---|
| **Accuracy** | % of all predictions that are correct |
| **Precision** | Of all websites flagged as phishing, how many actually are? |
| **Recall** | Of all actual phishing sites, how many did we catch? |
| **F1-Score** | Balance between Precision and Recall |
| **Confusion Matrix** | Shows exact counts of correct and wrong predictions |

### Cybersecurity Interpretation

- **False Negative (FN):** A phishing site is predicted as legitimate. **This is dangerous** — the user could be tricked.
- **False Positive (FP):** A legitimate site is predicted as phishing. **This is an inconvenience** — the user is incorrectly warned.
- In security applications, **high Recall** is often prioritised to minimise dangerous false negatives.

In [ ]:
# Generate predictions on test set
y_pred = rf_model.predict(X_test)
y_prob = rf_model.predict_proba(X_test)  # probability scores

In [ ]:
# ── Compute metrics ───────────────────────────────────────────────────────────
accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, pos_label=-1)  # phishing = positive class
recall    = recall_score(y_test, y_pred, pos_label=-1)
f1        = f1_score(y_test, y_pred, pos_label=-1)

print("=" * 45)
print(" MODEL EVALUATION RESULTS")
print("=" * 45)
print(f"  Accuracy  : {accuracy:.4f}  ({accuracy*100:.2f}%)")
print(f"  Precision : {precision:.4f}")
print(f"  Recall    : {recall:.4f}")
print(f"  F1-Score  : {f1:.4f}")
print("=" * 45)
print()
print("Interpretation (Positive class = Phishing):")
print(f"  • Precision {precision:.2%}: When the model says 'Phishing',")
print(f"    it is correct {precision:.0%} of the time.")
print(f"  • Recall {recall:.2%}: The model catches {recall:.0%} of all")
print(f"    actual phishing websites.")
print(f"  • F1 {f1:.2%}: Balanced score between precision and recall.")

In [ ]:
# ── Full Classification Report ────────────────────────────────────────────────
print("Full Classification Report:")
print("-" * 50)
print(classification_report(
    y_test, y_pred,
    target_names=['Phishing (-1)', 'Legitimate (1)'],
    digits=4
))

In [ ]:
# ── Confusion Matrix Visualisation ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))

cm = confusion_matrix(y_test, y_pred, labels=[-1, 1])
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['Phishing', 'Legitimate']
)
disp.plot(
    ax=ax,
    colorbar=True,
    cmap='Blues'
)
ax.set_title('Confusion Matrix — Random Forest', fontsize=14, fontweight='bold')

# Annotate with FP / FN / TP / TN
labels_annotation = [
    ['True Positives\n(Phishing detected correctly)', 'False Negatives\n(Phishing missed — DANGEROUS)'],
    ['False Positives\n(Legit flagged as phishing)', 'True Negatives\n(Legit detected correctly)']
]

plt.tight_layout()
plt.savefig('outputs/confusion_matrix.png', bbox_inches='tight', dpi=100)
plt.show()
print("Chart saved to outputs/confusion_matrix.png")

# Summarise the matrix in text
tn, fp, fn, tp = cm.ravel() if cm.shape == (2, 2) else (0, 0, 0, 0)
# Note: labels=[-1, 1] means row0=Phishing, col0=Phishing
# So: cm[0,0]=TP(Phishing), cm[0,1]=FN, cm[1,0]=FP, cm[1,1]=TN(Legit)
tp_ph, fn_ph, fp_ph, tn_ph = cm[0,0], cm[0,1], cm[1,0], cm[1,1]
print(f"\nConfusion Matrix Breakdown:")
print(f"  True  Positives (Phishing correctly detected) : {tp_ph:>5}")
print(f"  False Negatives (Phishing missed)             : {fn_ph:>5}  ← dangerous")
print(f"  False Positives (Legitimate wrongly flagged)  : {fp_ph:>5}")
print(f"  True  Negatives (Legitimate correctly passed) : {tn_ph:>5}")

In [ ]:
# ── Feature Importance ────────────────────────────────────────────────────────
importances = pd.Series(rf_model.feature_importances_, index=X.columns)
importances_sorted = importances.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 11))
colors_fi = ['#e74c3c' if v > 0.05 else '#3498db' for v in importances_sorted.values]
importances_sorted.plot(kind='barh', ax=ax, color=colors_fi, edgecolor='white')
ax.set_title('Random Forest — Feature Importances', fontsize=14, fontweight='bold')
ax.set_xlabel('Importance Score (Mean Decrease in Impurity)')
ax.axvline(0.05, color='orange', linestyle='--', linewidth=1, label='0.05 threshold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('outputs/feature_importance.png', bbox_inches='tight', dpi=100)
plt.show()
print("Chart saved to outputs/feature_importance.png")

print("\nTop 10 most important features for phishing detection:")
print(importances.sort_values(ascending=False).head(10).to_string())

## 12. Example Predictions on Test Samples

Let's look at actual test samples and see what the model predicts versus the true label.

In [ ]:
# Take 10 random test samples
np.random.seed(RANDOM_SEED)
sample_indices = np.random.choice(len(X_test), size=10, replace=False)

X_sample = X_test.iloc[sample_indices]
y_true_sample = y_test.iloc[sample_indices].values
y_pred_sample = rf_model.predict(X_sample)
y_prob_sample = rf_model.predict_proba(X_sample)

# Get class index for probability display
classes = list(rf_model.classes_)
phishing_idx = classes.index(-1)

print(f"{'#':<4} {'True Label':<18} {'Predicted':<18} {'Phishing Prob':<16} {'Match?'}")
print("-" * 75)

for i in range(len(y_true_sample)):
    true_lbl  = 'Phishing' if y_true_sample[i] == -1 else 'Legitimate'
    pred_lbl  = 'Phishing' if y_pred_sample[i] == -1 else 'Legitimate'
    ph_prob   = y_prob_sample[i][phishing_idx]
    match     = '✅' if y_true_sample[i] == y_pred_sample[i] else '❌'
    print(f"{i+1:<4} {true_lbl:<18} {pred_lbl:<18} {ph_prob:.4f}          {match}")

## 13. Save the Trained Model

We save the trained Random Forest model to the `models/` directory using `joblib`. This allows us to load and reuse the model later without retraining.

> **Note:** Always save the exact same version of the model used for evaluation to ensure reproducibility.

In [ ]:
os.makedirs('models', exist_ok=True)
model_path = os.path.join('models', 'phishing_rf_model.pkl')

joblib.dump(rf_model, model_path)
print(f"✅ Model saved to: {model_path}")

# Quick verification: reload and re-predict to confirm it saved correctly
loaded_model = joblib.load(model_path)
verify_pred = loaded_model.predict(X_test[:5])
verify_orig = rf_model.predict(X_test[:5])
assert (verify_pred == verify_orig).all(), "Model reload mismatch!"
print("✅ Model reload verified — predictions match original model.")

## 14. Results and Interpretation

### Summary of Model Performance

The Random Forest model was trained on the UCI Phishing Websites Dataset and evaluated on a held-out test set (20% of data).

**Key Observations:**

1. **High Accuracy (typically ~97%):** The model correctly classifies the vast majority of websites. This is expected because the dataset features are carefully engineered from known phishing indicators.

2. **Recall vs Precision Trade-off:** The model prioritises recall (catching as many phishing sites as possible) over precision. This is the right trade-off in a security context — a missed phishing site is more harmful than an unnecessary warning.

3. **Feature Importances:** Features related to SSL certificates (`SSLfinal_State`), domain age (`age_of_domain`), and anchor URL patterns (`URL_of_Anchor`) tend to rank as the most discriminative features. This aligns with known phishing tactics.

4. **Balanced Dataset:** The dataset is approximately balanced (~55% phishing, ~45% legitimate), so accuracy is a meaningful metric here.

### Cybersecurity Implications

- **False Negatives** (phishing sites missed) are the critical failure mode. Even with high recall, some phishing sites will evade detection.
- **False Positives** cause user frustration but are not harmful.
- This model should be used as **one layer** of a multi-layered security strategy — not as a standalone protection mechanism.
- Regular retraining with new phishing examples is needed as attacker tactics evolve.

## 15. Conclusion, Limitations, and Future Scope

### Conclusion

This project successfully demonstrated how machine learning — specifically a **Random Forest classifier** — can be applied to the cybersecurity problem of **phishing website detection**. Using 30 URL and web-page features from the UCI Phishing Websites Dataset, the model learned to distinguish between legitimate and phishing websites with high accuracy.

The project covered the full ML pipeline: data loading, inspection, cleaning, EDA, feature engineering, model training, evaluation, and persistence. It also highlighted the cybersecurity importance of minimising false negatives.

### Limitations

| Limitation | Detail |
|---|---|
| **Static dataset** | Model was trained on a dataset collected at a specific point in time. Phishing tactics evolve. |
| **No live URL scanning** | The model uses pre-extracted features — it does not visit or scan URLs in real time. |
| **Feature extraction not automated** | Real-world use requires extracting the 30 features from a live URL first. |
| **False negatives possible** | No classifier is perfect. Some phishing sites will not be detected. |
| **Zero-day phishing** | Brand-new phishing techniques not in training data will be missed. |
| **Dataset age** | The UCI dataset dates from ~2012 — modern phishing may use different techniques. |

### Future Scope

1. **Real-time feature extraction:** Build a URL feature extractor that fetches and analyses live websites.
2. **Deep learning models:** Experiment with LSTM or transformer models on raw URL strings.
3. **Browser extension:** Integrate the model into a browser plugin for real-time warnings.
4. **Continuous learning:** Retrain the model regularly with fresh phishing examples from threat-intelligence feeds.
5. **Explainability (XAI):** Use SHAP or LIME to explain per-prediction feature contributions to end users.
6. **Multi-class detection:** Extend to detect suspicious and malicious categories, not just binary classification.
7. **Ensemble with blacklists:** Combine ML predictions with known phishing blacklists (Google Safe Browsing API) for better coverage.

---

**End of Notebook**  
*Sakshi Kumari — IBM SkillsBuild Cyber Security 2026, BharatCares*